In [ ]:
%load_ext autoreload
%autoreload 2
# %matplotlib widget
%pdb off

from matplotlib import pyplot as plt
from matplotlib.lines import Line2D
import numpy as np
import pandas as pd
from IPython.display import display, HTML
import pyafn
from pyafn import rho, Cd
import random
from scipy.stats import lognorm
from scipy.stats import norm
from scipy.optimize import curve_fit
from emulationHelpers import readEmulationMI, plot_ventilation_model_fit, plot_empirical_model_error_distribution, average_realizations, fit_signed_model
from functools import partial
import seaborn as sns
from flowEmulationUtils import aggregate_window_series_to_room

#close all figures
plt.close('all')
plt.rcParams['figure.dpi'] = 140
im_scaling = .75
plt.rcParams['figure.figsize'] = [6.4 * im_scaling, 4.8 * im_scaling]
plt.rcParams['font.family'] = 'Times New Roman'

home_dir = "./"
display(home_dir)


In [ ]:
flowStatsMI, roomVentilationMI = readEmulationMI(home_dir=home_dir)

# Each window-only condition was simulated twice (two realizations, e.g. config2 R46 and R53). Every model is
# fit and run per realization, with that realization's own pressure-only pressures; metrics and plots then use
# the predicted and LES ventilation rates averaged over the realizations of each condition
# (emulationHelpers.average_realizations). False evaluates every realization separately.
averageRealizations = True
avg = partial(average_realizations, average=averageRealizations)
plot_ventilation_model_fit = partial(plot_ventilation_model_fit, average_over_realizations=averageRealizations)

y_var = "flux-Norm"
x_var = "p-noInt_optp0-q_model-Norm"
room_type_order = ["cross", "corner", "dual", "single"]
room_type_display_labels = {
    "cross": "Cross",
    "corner": "Corner",
    "dual": "Dual-Room",
    "single": "Single-Sided",
}
room_type_palette = dict(
    zip(room_type_order, sns.color_palette("colorblind", n_colors=len(room_type_order)))
)
sigma_q_label = "$\\sigma_{q_n}$ Bulk Fit PN"
sigma_p_label = "$\\sigma_{\\Delta C_p}$ Bulk Fit PN"
model_order = [sigma_q_label, sigma_p_label]
model_tick_labels = {
    sigma_q_label: "$\\sigma_{q_n}$",
    sigma_p_label: "$\\sigma_{\\Delta C_p}$",
}


In [ ]:
# Row 1: baseline q model (categorical hue)
fig, ax, xAdjusted, fittedParams = plot_ventilation_model_fit(
    data=flowStatsMI,
    y_var=y_var,
    x_var=x_var,
    x_var2=None,
    hue="roomType",
    style="slAll",
    model_func=pyafn.ventilationReDecomp_q,
    model_name="$\sigma_{q_n}$ Bulk Fit PN",
    p0=[1.0, 0.1],
    bounds=([0.1, 0], [np.inf, np.inf]),
    adjustData=True,
    show_assymptotes=True,
    add_numeric_colorbar=False,
    return_data=True,
    return_params=True
)

flowStatsMI["q_model-Norm-Adjusted"] = xAdjusted

## ASHRAE Ventilation Rates

In [ ]:
windowASHRAE = pd.read_csv(f"{home_dir}/ashrae_exports/windowASHRAE.csv")
data=flowStatsMI

ashrae_lookup = windowASHRAE.set_index(["windowType", "roomA", "C"])["ventilationRate"]
data["ASHRAE"] = data.apply(
    lambda row: ashrae_lookup.get((row["windowType"], row["roomA"], row["C"]), 999) if not row["slAll"] else np.nan, axis=1
)

plot_df = avg(data[data["slAll"] == False], extra={"xAdjusted": xAdjusted})

def annotate_directional_r_squared(ax, frame, target_col, pred_spec, sign_col="Sdelp", fontsize=11):
    if isinstance(pred_spec, str):
        prediction = frame[pred_spec]
    else:
        prediction = pd.Series(pred_spec).reindex(frame.index)

    metric_frame = pd.DataFrame(
        {"observed": frame[target_col], "predicted": prediction, "sign": frame[sign_col]}
    ).dropna()
    annotation_specs = [
        (metric_frame["sign"] > 0, "Flow in", (0.03, 0.97), "left", "top", "tab:red"),
        (metric_frame["sign"] < 0, "Flow out", (0.97, 0.03), "right", "bottom", "tab:blue"),
    ]
    for mask, label, position, horizontal_alignment, vertical_alignment, color in annotation_specs:
        subset = metric_frame.loc[mask]
        denominator = np.sum((subset["observed"] - subset["observed"].mean()) ** 2)
        if subset.empty or denominator <= 0:
            continue
        residual = subset["predicted"] - subset["observed"]
        r_squared = 1 - np.sum(residual ** 2) / denominator
        ax.text(
            *position,
            rf"{label}: $R^2={r_squared:.2f}$",
            transform=ax.transAxes,
            ha=horizontal_alignment,
            va=vertical_alignment,
            fontsize=fontsize,
            color=color,
            bbox={"facecolor": "white", "edgecolor": "none", "alpha": 0.75, "pad": 1.5},
            zorder=5,
        )

fig, axs = plt.subplots(1, 3, figsize=(12, 4.8), dpi=160, sharey=True)

scatter_specs = [
    ("ASHRAE", "ASHRAE Pressures",  "Modeled Flux Velocity"),
    (x_var, "LES Pressures", "Modeled Flux Velocity"),
    ("xAdjusted", "LES Pressures", "Fluctuation-Adjusted Flux Velocity"),
]

for ax, (xcol, title, xlabel) in zip(axs, scatter_specs):
    if type(xcol) == str:
        print(f"{xcol}: {len(plot_df[xcol].dropna())}")
    sns.scatterplot(
        data=plot_df,
        x=xcol,
        y=y_var,
        hue="roomType",
        # style="roomType",
        palette="colorblind",
        hue_order=room_type_order,
        alpha=0.65,
        s=20,
        ax=ax,
        legend=ax is axs[0],
    )

    lim_min = np.nanmin([plot_df[x_var].min(), plot_df[y_var].min()])
    lim_max = np.nanmax([plot_df[x_var].max(), plot_df[y_var].max()])
    ax.plot(
        [lim_min, lim_max],
        [lim_min, lim_max],
        "r--",
        linewidth=1.2,
        label="1:1" if ax is axs[0] else None,
    )

    ax.set_title(title, fontsize=18)
    ax.set_xlabel(xlabel, fontsize=16)
    ax.set_ylabel("LES Flux Velocity", fontsize=16)
    ax.grid(True, alpha=0.3)
    ax.tick_params(labelsize=14)
    annotate_directional_r_squared(ax, plot_df, y_var, xcol, fontsize=11)
    # ax.set_xlim(-0.6, 0.6)
    # ax.set_ylim(-0.6, 0.6)


room_handles = [
    Line2D(
        [0], [0],
        color=room_type_palette[room_type],
        marker="o",
        linestyle="None",
        markersize=7,
        label=room_type_display_labels[room_type],
    )
    for room_type in room_type_order
]
for ax in axs:
    if ax.get_legend() is not None:
        ax.get_legend().remove()

fig.legend(
    room_handles,
    [handle.get_label() for handle in room_handles],
    loc="lower center",
    bbox_to_anchor=(0.5, 0.01),
    ncol=len(room_handles),
    fontsize=11,
    frameon=False,
)

fig.suptitle("Window-Level Ventilation Comparison", fontsize=20)
fig.subplots_adjust(left=0.08, right=0.98, top=0.83, bottom=0.26, wspace=0.30)

In [ ]:
data_WA_mean = data[data["slAll"] == False].groupby(["windowType", "roomA"])[y_var].mean().reset_index()

windowTypeOrder = data_WA_mean["windowType"].dropna().unique()
plt.figure()
sns.lineplot(data=data_WA_mean, x="roomA", y=y_var, hue="windowType", palette="tab10", hue_order=windowTypeOrder)
sns.lineplot(data=windowASHRAE, x="roomA", y="ventilationRate", hue="windowType", palette="tab10", linestyle='--', hue_order=windowTypeOrder)

In [ ]:
roomVentilationMI_adj = aggregate_window_series_to_room(
    roomVentilationMI, xAdjusted, out_col="xAdjusted_room", mode="abs_half_sum"
)

In [ ]:
roomASHRAE = pd.read_csv(f"{home_dir}/ashrae_exports/roomASHRAE.csv")

ashrae_lookup = roomASHRAE.set_index(["roomType", "AofA", "C"])["ventilationRate"]
roomVentilationMI_adj["ASHRAE"] = roomVentilationMI_adj.apply(
    lambda row: ashrae_lookup.get((row["roomType"], row["AofA"], row["C"]), np.nan) if not row["slAll"] else np.nan, axis=1
)

plt.figure()
sns.scatterplot(data=avg(roomVentilationMI_adj[roomVentilationMI_adj["slAll"] == False]), x="ASHRAE", y=y_var, hue="AofA", palette="viridis")

## ASHRAE PN Bulk-Fit Comparison

In [ ]:
def attach_ashrae(frame, ashrae_frame, keys):
    lookup = ashrae_frame.set_index(keys)["ventilationRate"]
    out = frame.copy()
    out["ASHRAE"] = out.apply(
        lambda row: lookup.get(tuple(row[key] for key in keys), np.nan), axis=1
    )
    if "slAll" in out.columns:
        out.loc[out["slAll"], "ASHRAE"] = np.nan
    return out


def signed_group_specs(frame, sign_col="Sdelp"):
    return [
        ("group", "Flow Entering", frame[sign_col] > 0),
        ("group", "Flow Exiting", frame[sign_col] < 0),
    ]


def compute_error_metrics(frame, target_col, pred_col, model_name, *, group_col=None, group_specs=None):
    if group_specs is None:
        group_specs = [("group", group, frame[group_col] == group) for group in frame[group_col].dropna().unique()]

    rows = []
    for scope, group, mask in [("total", "All", pd.Series(True, index=frame.index)), *group_specs]:
        subset = frame.loc[mask, [target_col, pred_col]].dropna()
        if subset.empty:
            continue
        error = subset[pred_col] - subset[target_col]
        rmse = np.sqrt(np.mean(error**2))
        target_std = np.std(subset[target_col])
        rows.append(
            {
                "model_name": model_name,
                "scope": scope,
                "group": group,
                "rmse": rmse,
                "bias": error.mean(),
                "iqr_error": error.quantile(0.75) - error.quantile(0.25),
                "nrmse": rmse / target_std if target_std > 0 else np.nan,
                "n": len(subset),
            }
        )
    return pd.DataFrame(rows)


def fit_signed_prediction(frame, x_col, y_col, model_func, p0, bounds, *, out_col, model_name, sign_col="Sdelp", min_points=6, fixed_sigma=None, fit_source="ASHRAE-to-LES Fit"):
    # Per flow direction s, q = s * model(|x|). With averageRealizations the parameters minimize the error of the
    # predictions averaged over the realizations of each condition, the error reported (fit_signed_model)
    funcs, inits = {}, {}
    for s in (1, -1):
        # fixed_sigma: {sign: sigma} pins the fluctuation scale so only the Cd scaling is fit
        if fixed_sigma is None:
            funcs[s], inits[s] = model_func, (p0, bounds)
        else:
            funcs[s] = lambda x, a, sigma=fixed_sigma[s]: model_func(x, a, sigma)
            inits[s] = (p0[:1], (bounds[0][:1], bounds[1][:1]))
    valid = frame[x_col].notna() & frame[y_col].notna()
    popts = fit_signed_model(frame[valid], x_col, y_col, funcs, inits, sign_col=sign_col,
                             average=averageRealizations, min_points=min_points, label=model_name)

    prediction = pd.Series(np.nan, index=frame.index, dtype=float)
    rows = []

    for s, direction in [(1, "Flow Entering"), (-1, "Flow Exiting")]:
        if s not in popts:
            continue
        popt, fit_func = popts[s], funcs[s]
        mask = (frame[sign_col] == s) & valid
        regdf = frame.loc[mask]
        prediction.loc[mask] = fit_func(frame.loc[mask, x_col].abs().to_numpy(), *popt) * s
        rows.append(
            {
                "fit_source": fit_source,
                "model_name": model_name,
                "opening_group": "Window",
                "direction": direction,
                "cd": Cd * popt[0],
                "sigma": popt[1] if len(popt) > 1 else (fixed_sigma[s] if fixed_sigma is not None else np.nan),
                "n": len(regdf),
                "x_label": x_col,
                "y_label": y_col,
            }
        )

    return prediction.rename(out_col), pd.DataFrame(rows)


def annotate_total_r_squared(ax, frame, target_col, pred_col, fontsize=11):
    subset = frame[[target_col, pred_col]].dropna()
    denominator = np.sum((subset[target_col] - subset[target_col].mean()) ** 2)
    if subset.empty or denominator <= 0:
        return
    r_squared = 1 - np.sum((subset[pred_col] - subset[target_col]) ** 2) / denominator
    ax.text(
        0.03,
        0.97,
        rf"$R^2={r_squared:.2f}$",
        transform=ax.transAxes,
        ha="left",
        va="top",
        fontsize=fontsize,
        color="black",
        bbox={"facecolor": "white", "edgecolor": "none", "alpha": 0.75, "pad": 1.5},
        zorder=5,
    )


def plot_model_scatter(ax, frame, target_col, pred_col, *, hue_col, palette, hue_order, title, x_label="Prediction", annotate_directions=False, annotate_total=False, annotation_fontsize=10, lims=None):
    sns.scatterplot(
        data=frame,
        x=pred_col,
        y=target_col,
        hue=hue_col,
        palette=palette,
        hue_order=hue_order,
        alpha=0.72,
        s=12,
        ax=ax,
        legend=False,
    )
    vals = pd.concat([frame[target_col], frame[pred_col]]).dropna()
    lo, hi = (vals.min(), vals.max()) if lims is None else lims
    ax.plot([lo, hi], [lo, hi], "r--", linewidth=1.2)
    ax.set_xlim(lo, hi)
    ax.set_ylim(lo, hi)
    ax.set_title(title, fontsize=15)
    ax.set_xlabel(x_label, fontsize=13)
    ax.set_ylabel("LES Flux Velocity", fontsize=13)
    ax.grid(True, alpha=0.3)
    ax.tick_params(labelsize=11)
    if annotate_directions:
        annotate_directional_r_squared(ax, frame, target_col, pred_col, fontsize=annotation_fontsize)
    if annotate_total:
        annotate_total_r_squared(ax, frame, target_col, pred_col, fontsize=annotation_fontsize)


def plot_error_summary(metrics_df, palette, model_order, model_tick_labels, title, group_label_map=None):
    group_label_map = group_label_map or {}
    metric_rows = [("rmse", "RMSE"), ("bias", "Bias"), ("iqr_error", "Error IQR")]
    fig, axs = plt.subplots(len(metric_rows), 1, figsize=(7.8, 5.8), dpi=160, sharey=True)
    group_data = metrics_df[metrics_df["scope"] == "group"].copy()
    group_order = [group for group in palette if group in group_data["group"].unique()]

    for ax, (metric_col, metric_label) in zip(axs, metric_rows):
        for model_idx, model_name in enumerate(model_order):
            values = []
            for group in group_order:
                row = group_data[(group_data["model_name"] == model_name) & (group_data["group"] == group)]
                if row.empty:
                    continue
                value = row.iloc[0][metric_col]
                values.append(value)
                ax.scatter(value, model_idx, color=palette[group], edgecolor="black", linewidth=0.5, s=38, zorder=3)
            if len(values) > 1:
                ax.plot(values, [model_idx] * len(values), color="0.35", alpha=0.25, linewidth=1.0, zorder=2)
        ax.axvline(0, color="k", linewidth=0.8, alpha=0.7)
        ax.grid(axis="x", alpha=0.25)
        ax.grid(axis="y", alpha=0.35, linestyle="--")
        ax.set_yticks(range(len(model_order)))
        ax.set_yticklabels([model_tick_labels[name] for name in model_order], fontsize=12)
        ax.set_xlabel(metric_label, fontsize=13)
        ax.tick_params(axis="x", labelsize=11)

    handles = [
        Line2D(
            [0], [0],
            color=color,
            marker="o",
            linestyle="None",
            markersize=7,
            label=group_label_map.get(group, group),
        )
        for group, color in palette.items()
        if group in group_order
    ]
    if handles:
        fig.legend(handles, [h.get_label() for h in handles], loc="lower center", ncol=len(handles), frameon=False, fontsize=10)
    fig.suptitle(title, fontsize=16)
    fig.tight_layout(rect=[0, 0.08, 1, 0.95])
    return fig, axs


In [ ]:
raw_ashrae_label = "ASHRAE Mean-Pressure Baseline"
ashrae_q_label = "ASHRAE $\\sigma_{q_n}$ Bulk Fit"
ashrae_p_label = "ASHRAE $\\sigma_{\\Delta C_p}$ Bulk Fit"
fit_bounds = ([0.1, 0], [np.inf, np.inf])

fit_specs = [
    {
        "model_name": ashrae_q_label,
        "tick_label": "ASHRAE $\\sigma_{q_n}$ Bulk Fit",
        "pred_col": "ashrae_sigma_q_fit",
        "room_col": "ashrae_sigma_q_fit_room",
        "model_func": pyafn.ventilationReDecomp_q,
        "p0": [1.0, 0.1],
    },
    {
        "model_name": ashrae_p_label,
        "tick_label": "ASHRAE $\\sigma_{\\Delta C_p}$ Bulk Fit",
        "pred_col": "ashrae_sigma_p_fit",
        "room_col": "ashrae_sigma_p_fit_room",
        "model_func": pyafn.ventilationReDecomp_p,
        "p0": [1.0, 0.01],
    },
]

window_plot_df = attach_ashrae(flowStatsMI, windowASHRAE, ["windowType", "roomA", "C"])
window_plot_df = window_plot_df[window_plot_df["slAll"] == False].copy()

fit_param_frames = []
for spec in fit_specs:
    window_plot_df[spec["pred_col"]], params_df = fit_signed_prediction(
        window_plot_df,
        "ASHRAE",
        y_var,
        spec["model_func"],
        spec["p0"],
        fit_bounds,
        out_col=spec["pred_col"],
        model_name=spec["model_name"],
    )
    fit_param_frames.append(params_df)

room_plot_df = roomVentilationMI.copy()
for spec in fit_specs:
    room_plot_df = aggregate_window_series_to_room(
        room_plot_df,
        window_plot_df[spec["pred_col"]],
        out_col=spec["room_col"],
        mode="abs_half_sum",
    )
room_plot_df = attach_ashrae(room_plot_df, roomASHRAE, ["roomType", "AofA", "C"])
room_plot_df = room_plot_df[room_plot_df["slAll"] == False].copy()

# Metrics and plots use the predictions and LES rates averaged over the realizations of each condition
window_eval_df = avg(window_plot_df)
room_eval_df = avg(room_plot_df)

model_specs_for_metrics = [
    {"model_name": raw_ashrae_label, "tick_label": "ASHRAE Mean-Pressure Baseline", "window_col": "ASHRAE", "room_col": "ASHRAE", "evaluation": "raw_ashrae"},
    *[{"model_name": spec["model_name"], "tick_label": spec["tick_label"], "window_col": spec["pred_col"], "room_col": spec["room_col"], "evaluation": "ashrae_to_les_fit"} for spec in fit_specs],
]
model_order = [spec["model_name"] for spec in model_specs_for_metrics]
model_tick_labels = {spec["model_name"]: spec["tick_label"] for spec in model_specs_for_metrics}

window_flow_groups = signed_group_specs(window_eval_df)
window_palette = {"Flow Entering": "red", "Flow Exiting": "blue"}
room_palette = {
    room_type: color
    for room_type, color in zip(room_type_order, sns.color_palette("colorblind", n_colors=len(room_type_order)))
    if room_type in room_plot_df["roomType"].dropna().unique()
}

window_error_metrics_df = pd.concat(
    [compute_error_metrics(window_eval_df, y_var, spec["window_col"], spec["model_name"], group_specs=window_flow_groups).assign(dataset="windows", evaluation=spec["evaluation"]) for spec in model_specs_for_metrics],
    ignore_index=True,
)
room_error_metrics_df = pd.concat(
    [compute_error_metrics(room_eval_df, y_var, spec["room_col"], spec["model_name"], group_col="roomType").assign(dataset="rooms", evaluation=spec["evaluation"]) for spec in model_specs_for_metrics],
    ignore_index=True,
)
ashrae_error_metrics_export_df = pd.concat([window_error_metrics_df, room_error_metrics_df], ignore_index=True)[
    ["dataset", "evaluation", "model_name", "scope", "group", "rmse", "bias", "iqr_error", "nrmse", "n"]
]
ashrae_fit_params_export_df = pd.concat(fit_param_frames, ignore_index=True)[
    ["fit_source", "model_name", "opening_group", "direction", "cd", "sigma", "n", "x_label", "y_label"]
]

window_match_summary = pd.DataFrame({"dataset": ["windows"], "rows": [len(window_eval_df)], "ashrae_matches": [window_eval_df["ASHRAE"].notna().sum()]})
room_match_summary = pd.DataFrame({"dataset": ["rooms"], "rows": [len(room_eval_df)], "ashrae_matches": [room_eval_df["ASHRAE"].notna().sum()]})


In [ ]:
plot_specs = [("ASHRAE", raw_ashrae_label), *[(spec["pred_col"], spec["model_name"]) for spec in fit_specs]]
room_plot_specs = [(next(spec["room_col"] for spec in model_specs_for_metrics if spec["model_name"] == label), label) for _, label in plot_specs]
hue_order = [room for room in room_type_order if room in room_palette]

fig_fit, axs_fit = plt.subplots(2, len(plot_specs), figsize=(10.0, 9.0), sharey="row", sharex="row", dpi=160)
for col_idx, ((window_col, label), (room_col, _)) in enumerate(zip(plot_specs, room_plot_specs)):
    plot_model_scatter(
        axs_fit[0, col_idx],
        window_eval_df,
        y_var,
        window_col,
        hue_col="roomType",
        palette=room_palette,
        hue_order=hue_order,
        title="",
        x_label=model_tick_labels[label],
        annotate_directions=True,
    )
    plot_model_scatter(
        axs_fit[1, col_idx],
        room_eval_df,
        y_var,
        room_col,
        hue_col="roomType",
        palette=room_palette,
        hue_order=hue_order,
        title="",
        x_label=model_tick_labels[label],
        annotate_total=True,
    )

for ax in axs_fit.ravel():
    ax.set_ylabel("")
    ax.set_xlabel("")

for col_idx, (_, label) in enumerate(plot_specs):
    axs_fit[0, col_idx].set_title(model_tick_labels[label], fontsize=14, pad=6)

for row_idx, row_label in enumerate(["Windows", "Rooms"]):
    axs_fit[row_idx, 0].set_ylabel(row_label, fontsize=14, labelpad=10)

room_handles_fit = [
    Line2D(
        [0], [0],
        color=room_palette[room_type],
        marker="o",
        linestyle="None",
        markersize=7,
        label=room_type_display_labels[room_type],
    )
    for room_type in hue_order
]
axs_fit[1, 0].legend(
    room_handles_fit,
    [handle.get_label() for handle in room_handles_fit],
    loc="lower right",
    frameon=True,
    fontsize=10,
)

fig_fit.supxlabel("$q_{PW,n}$", fontsize=16, y=0.10, x=0.54)
fig_fit.supylabel("LES $\overline{q_n}$", fontsize=16, x=0.00, y=0.51)
# fig_fit.suptitle("ASHRAE Fits Evaluated Against LES", fontsize=18)
fig_fit.subplots_adjust(left=0.10, right=0.98, top=0.86, bottom=0.16, wspace=0.24, hspace=0.28)

fig_window_error, axs_window_error = plot_error_summary(
    window_error_metrics_df,
    window_palette,
    model_order,
    model_tick_labels,
    "Window ASHRAE Prediction Errors Relative to LES",
)
fig_room_error, axs_room_error = plot_error_summary(
    room_error_metrics_df,
    room_palette,
    model_order,
    model_tick_labels,
    "Room ASHRAE Prediction Errors Relative to LES",
    group_label_map=room_type_display_labels,
)


In [ ]:
# Same 3x2 layout, but columns 2-3 are the LES PN Bulk Measured models driven by LES pressures (as in the logNorm notebook):
# sigma fixed to the LES mean RMS per flow direction, Cd fit on all window rows (incl. skylight configs)
measured_specs = [
    {
        "model_name": "LES $\\sigma_{q_n}$ PN Bulk Measured",
        "tick_label": "LES $\\sigma_{q_n}$ PN Bulk",
        "pred_col": "les_sigma_q_measured",
        "room_col": "les_sigma_q_measured_room",
        "model_func": pyafn.ventilationReDecomp_q,
        "sigma_col": "rms-mass_flux-Norm",
    },
    {
        "model_name": "LES $\\sigma_{\\Delta C_p}$ PN Bulk Measured",
        "tick_label": "LES $\\sigma_{\\Delta C_p}$ PN Bulk",
        "pred_col": "les_sigma_p_measured",
        "room_col": "les_sigma_p_measured_room",
        "model_func": pyafn.ventilationReDecomp_p,
        "sigma_col": "p_rms-noInt-Norm",
    },
]

les_window_fit_df = flowStatsMI[flowStatsMI["skylight"] == 0]
measured_param_frames = []
for spec in measured_specs:
    sigma_by_sign = avg(les_window_fit_df).groupby("Sdelp")[spec["sigma_col"]].mean().to_dict()
    les_prediction, params_df = fit_signed_prediction(
        les_window_fit_df,
        x_var,
        y_var,
        spec["model_func"],
        [1.0],
        fit_bounds,
        out_col=spec["pred_col"],
        model_name=spec["model_name"],
        fixed_sigma=sigma_by_sign,
        fit_source="LES PN Bulk Measured",
    )
    measured_param_frames.append(params_df)
    window_plot_df[spec["pred_col"]] = les_prediction.reindex(window_plot_df.index)
    room_plot_df = aggregate_window_series_to_room(
        room_plot_df,
        les_prediction,
        out_col=spec["room_col"],
        mode="abs_half_sum",
    )
    model_tick_labels[spec["model_name"]] = spec["tick_label"]
display(pd.concat(measured_param_frames, ignore_index=True))
window_eval_df = avg(window_plot_df)
room_eval_df = avg(room_plot_df)

measured_plot_specs = [("ASHRAE", "ASHRAE", raw_ashrae_label), *[(spec["pred_col"], spec["room_col"], spec["model_name"]) for spec in measured_specs]]

fig_meas, axs_meas = plt.subplots(2, len(measured_plot_specs), figsize=(10.0, 9.0), sharey="row", sharex="row", dpi=160)
for col_idx, (window_col, room_col, label) in enumerate(measured_plot_specs):
    plot_model_scatter(
        axs_meas[0, col_idx],
        window_eval_df,
        y_var,
        window_col,
        hue_col="roomType",
        palette=room_palette,
        hue_order=hue_order,
        title="",
        annotate_directions=True,
    )
    plot_model_scatter(
        axs_meas[1, col_idx],
        room_eval_df,
        y_var,
        room_col,
        hue_col="roomType",
        palette=room_palette,
        hue_order=hue_order,
        title="",
        annotate_total=True,
    )
    axs_meas[0, col_idx].set_title(model_tick_labels[label], fontsize=14, pad=6)

for ax in axs_meas.ravel():
    ax.set_ylabel("")
    ax.set_xlabel("")

for row_idx, row_label in enumerate(["Windows", "Rooms"]):
    axs_meas[row_idx, 0].set_ylabel(row_label, fontsize=14, labelpad=10)

axs_meas[1, 0].legend(
    room_handles_fit,
    [handle.get_label() for handle in room_handles_fit],
    loc="lower right",
    frameon=True,
    fontsize=10,
)

fig_meas.supxlabel("$q_{PW,n}$", fontsize=16, y=0.10, x=0.54)
fig_meas.supylabel("LES $\overline{q_n}$", fontsize=16, x=0.00, y=0.51)
fig_meas.subplots_adjust(left=0.10, right=0.98, top=0.86, bottom=0.16, wspace=0.24, hspace=0.28)

In [ ]:
# 2x2 baseline comparison: ASHRAE vs LES mean-pressure baseline (Cd = pyafn.Cd, sigma = 0, i.e. the q model from LES pressures)
les_baseline_label = "LES Mean-Pressure Baseline"
model_tick_labels[les_baseline_label] = les_baseline_label
baseline_plot_specs = [("ASHRAE", "ASHRAE", raw_ashrae_label), (x_var, x_var, les_baseline_label)]

fig_base, axs_base = plt.subplots(2, len(baseline_plot_specs), figsize=(9.6, 8), sharey="row", sharex="row", dpi=160, layout="constrained")
for col_idx, (window_col, room_col, label) in enumerate(baseline_plot_specs):
    plot_model_scatter(
        axs_base[0, col_idx],
        window_eval_df,
        y_var,
        window_col,
        hue_col="roomType",
        palette=room_palette,
        hue_order=hue_order,
        title="",
        annotate_directions=True,
        annotation_fontsize=14,
        lims=(-0.62, 0.62),
    )
    plot_model_scatter(
        axs_base[1, col_idx],
        room_eval_df,
        y_var,
        room_col,
        hue_col="roomType",
        palette=room_palette,
        hue_order=hue_order,
        title="",
        annotate_total=True,
        annotation_fontsize=14,
        lims=(0.0, 0.62),
    )
    axs_base[0, col_idx].set_title(model_tick_labels[label], fontsize=18, pad=10)

for ax in axs_base[0]:
    ax.set_xticks(np.arange(-0.6, 0.61, 0.2))
    ax.set_yticks(np.arange(-0.6, 0.61, 0.2))
for ax in axs_base[1]:
    ax.set_xticks(np.arange(0.0, 0.61, 0.2))
    ax.set_yticks(np.arange(0.0, 0.61, 0.2))

for ax in axs_base.ravel():
    ax.set_ylabel("")
    ax.set_xlabel("")
    ax.tick_params(labelsize=16)

for row_idx, row_label in enumerate(["Windows", "Rooms"]):
    axs_base[row_idx, 0].set_ylabel(row_label, fontsize=18)

# Top-left panel, just below its "Flow in" R^2 label
axs_base[0, 0].legend(
    room_handles_fit,
    [handle.get_label() for handle in room_handles_fit],
    loc="upper left",
    bbox_to_anchor=(0.0, 0.89),
    fontsize=14, frameon=True, facecolor="white", edgecolor="0.6", framealpha=0.9,
    fancybox=False, handletextpad=0.3, borderpad=0.3, labelspacing=0.3,
)

fig_base.supxlabel("$q_{MP,n}$", fontsize=24)
fig_base.supylabel("LES $\overline{q_n}$", fontsize=24)

In [ ]:
import os

ashrae_export_dir = os.path.join(home_dir, "ashrae_exports")
os.makedirs(ashrae_export_dir, exist_ok=True)

ashrae_export_tables = {
    "ashrae_error_metrics.csv": ashrae_error_metrics_export_df,
    "ashrae_fit_params.csv": ashrae_fit_params_export_df,
}
ashrae_export_paths = {filename: os.path.join(ashrae_export_dir, filename) for filename in ashrae_export_tables}
for filename, df in ashrae_export_tables.items():
    df.to_csv(ashrae_export_paths[filename], index=False)

ashrae_export_manifest_df = pd.DataFrame(
    {
        "table_name": list(ashrae_export_tables.keys()),
        "rows": [len(df) for df in ashrae_export_tables.values()],
        "path": [ashrae_export_paths[filename] for filename in ashrae_export_tables],
    }
)

display(window_match_summary)
display(room_match_summary)
display(ashrae_error_metrics_export_df)
display(ashrae_fit_params_export_df)
display(ashrae_export_manifest_df)
